# 06 小型Set/Spectrum Transformer直接预测PBT相关性质

本步骤只使用冻结的20 ppm Development公共谱图。ChemVista继续锁定。模型不使用SMILES、分子指纹或候选结构作为输入；结构身份只用于离线去重、无标签同结构谱图配对及结构簇fold隔离。

训练路线包括：预训练完整模型、无预训练模型、去neutral loss消融和去碰撞能消融。无标签预训练背景集排除全部Development与冻结ChemVista评价结构。

In [ ]:
from pathlib import Path
import sys
import yaml
import torch
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'configs' / 'default.yaml').exists():
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / 'configs' / 'default.yaml').exists()), ROOT)
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbt_msms.transformer_pipeline import MAIN_ROUTE, run_pipeline

with (ROOT / 'configs' / 'default.yaml').open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)
MODEL_CONFIG = CONFIG['models']['transformer']
print('ROOT:', ROOT)
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('模型配置:', MODEL_CONFIG)

## 预训练、五折微调与消融

预训练同时优化遮蔽峰重建和同结构不同谱图的对比损失。外层测试fold从不参与模型训练；每个外层训练集再保留一个完整结构簇fold用于选择训练轮数，随后在完整外层训练集重训。

In [ ]:
result = run_pipeline(ROOT, MODEL_CONFIG)
print('输出目录:', result['output_dir'])
display(result['pretrain_history'])
display(result['selection'])

## 化合物级内部OOF结果

回归任务以MAE为主指标，同时完整报告RMSE、R²和Spearman及化合物级bootstrap区间；P筛查以PR-AUC为主指标。以下均为同一化合物全部合格谱图预测中位数。

In [ ]:
metrics = result['metrics'].copy()
main_metrics = metrics[metrics['representation_route'].eq(MAIN_ROUTE)]
display(main_metrics[['task', 'kind', 'n_compounds', 'primary_metric', 'primary_value', 'primary_ci_low', 'primary_ci_high', 'mae', 'rmse', 'r2', 'spearman', 'pr_auc']])
display(metrics[['representation_route', 'task', 'primary_metric', 'primary_value', 'r2', 'spearman']].sort_values(['task', 'representation_route']))
display(result['comparisons'])

## 验收与外测锁定

全部检查必须PASS才会写出最终运行清单。当前步骤不产生ChemVista性能指标。

In [ ]:
display(result['qc'])
print('参数量:', result['manifest']['architecture']['parameter_count'])
print('峰顺序置换最大预测差:', result['manifest']['permutation_invariance_max_abs_difference'])
print('ChemVista unlocked:', result['manifest']['external_test_unlocked'])
assert result['qc']['status'].eq('PASS').all()
assert result['manifest']['external_test_unlocked'] is False